In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pickle
import seaborn as sns
import torch
import config
from utils.helpers import get_max
from models import SingleTransformer
from train import train_mlm, train_cls
from evaluate import evaluate_mlm, evaluate_cls_cv
from data import create_dataset, load_data
from interpretation.visualization import *


In [ ]:
adata_RNA_labelled, adata_RNA_unlabelled, df_degs = load_data.load_processed_rna(verbose=True)

print("RNA data loaded.",
        "\nRNA D3 Labelled:", adata_RNA_labelled.shape, adata_RNA_labelled.obs.shape,  list(adata_RNA_labelled.var_names)[:5],"...",
        "\nRNA D3 Un-Labelled:", adata_RNA_unlabelled.shape, adata_RNA_unlabelled.obs.shape, list(adata_RNA_unlabelled.var_names)[:5],"...")

RNA data loaded. 
RNA D3 Labelled: (2008, 944) (2008, 16) ['Upk1b', '2010300F17Rik', 'Tspan13', 'Tnmd', 'F2rl1'] ... 
RNA D3 Un-Labelled: (100254, 944) (100254, 8) ['Upk1b', '2010300F17Rik', 'Tspan13', 'Tnmd', 'F2rl1'] ...


In [ ]:
mlm_train_loader, mlm_val_loader = create_dataset.get_mlm_loaders(train_data=adata_RNA_unlabelled, 
                                                                  val_data=adata_RNA_labelled, 
                                                                  batch_size=32, 
                                                                  batch_key="batch_no", 
                                                                  data_dtype=torch.int32)

labelled_dataset, pcts, gene_names = create_dataset.get_cls_dataset(data=adata_RNA_labelled,
                                                    batch_key="batch_no",
                                                    label_key="label",
                                                    pct_key="pct",
                                                    filter_pcts=20.0,
                                                    data_dtype=torch.int32)
print(len(labelled_dataset))

In [7]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

model_config = {
    "vocab_size": int(get_max([adata_RNA_labelled, adata_RNA_unlabelled])) + 2,
    "seq_len": next(iter(mlm_train_loader))[0].shape[-1],
    "d_model": 128,
    # "d_batch": 4,
    "d_ff": 16,
    "n_heads": 8,
    "n_encoder_layers": 2,
    "n_batches": 3,
    "dropout_rate": 0.2
}
model = SingleTransformer(id="RNA", **model_config).to(device)

In [ ]:
mlm_train_loss, mlm_val_loss = train_mlm(model, mlm_train_loader, mlm_val_loader, device,
                                            mse_based=False, epochs=10, lr=1e-3, weight_decay=0, tune_lr=True, 
                                            save_folder="ckp/MLM/", id="RNA", use_multiple_gpu=False)
plot_training_vs_validation_losses(mlm_train_loss, mlm_val_loss)

# Note
**Update path for MLM checkpoint in the `config.py` file for `MLM_RNA_CKP`**

In [ ]:
model.load_state_dict(torch.load(config.MLM_RNA_CKP))
print(f"Model loaded from checkpoint: {config.MLM_RNA_CKP}")
val_loss = evaluate_mlm(model, mlm_val_loader, mask_token=-1, mse_based=False, device=device)
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
fold_results = train_cls("RNA", model_config, 
                        labelled_dataset, k_folds=5, batch_size=16, 
                        epochs=15, lr=5e-4, weight_decay=1e-3,
                        use_mlm=True, mlm_path=config.MLM_RNA_CKP, save_path="ckp/CLS", 
                        device=device, loss_fn="w_bce", seed=42)

print(f"Cross-Validation Results:\n"
    f"Train AUC: {np.mean([fr['train_auc'] for fr in fold_results]):.4f} ± {np.std([fr['train_auc'] for fr in fold_results]):.4f}, "
    f"Valid AUC: {np.mean([fr['best_val_auc'] for fr in fold_results]):.4f} ± {np.std([fr['best_val_auc'] for fr in fold_results]):.4f}, "
    f"Precision: {np.mean([fr['best_precision'] for fr in fold_results]):.4f} ± {np.std([fr['best_precision'] for fr in fold_results]):.4f}, "
    f"Recall: {np.mean([fr['best_recall'] for fr in fold_results]):.4f} ± {np.std([fr['best_recall'] for fr in fold_results]):.4f},\n "
    f"F1: {np.mean([fr['best_f1'] for fr in fold_results]):.4f} ± {np.std([fr['best_f1'] for fr in fold_results]):.4f}, "
    f"Accuracy: {np.mean([fr['best_accuracy'] for fr in fold_results]):.4f} ± {np.std([fr['best_accuracy'] for fr in fold_results]):.4f}, "
    f"Specificity: {np.mean([fr['best_specificity'] for fr in fold_results]):.4f} ± {np.std([fr['best_specificity'] for fr in fold_results]):.4f}")

In [ ]:
fold_results_nomlm = train_cls("RNA", model_config, 
                            labelled_dataset, k_folds=5, batch_size=32, 
                            epochs=10, lr=1e-4, weight_decay=1e-4,
                            use_mlm=False, mlm_path=None, save_path="ckp/CLS", 
                            device=device, loss_fn="w_bce", seed=0)

print(f"Cross-Validation Results:\n"
    f"Train AUC: {np.mean([fr['train_auc'] for fr in fold_results_nomlm]):.4f} ± {np.std([fr['train_auc'] for fr in fold_results_nomlm]):.4f}, "
    f"Valid AUC: {np.mean([fr['best_val_auc'] for fr in fold_results_nomlm]):.4f} ± {np.std([fr['best_val_auc'] for fr in fold_results_nomlm]):.4f}, "
    f"Precision: {np.mean([fr['best_precision'] for fr in fold_results_nomlm]):.4f} ± {np.std([fr['best_precision'] for fr in fold_results_nomlm]):.4f}, "
    f"Recall: {np.mean([fr['best_recall'] for fr in fold_results_nomlm]):.4f} ± {np.std([fr['best_recall'] for fr in fold_results_nomlm]):.4f}, "
    f"F1: {np.mean([fr['best_f1'] for fr in fold_results_nomlm]):.4f} ± {np.std([fr['best_f1'] for fr in fold_results_nomlm]):.4f}, "
    f"Accuracy: {np.mean([fr['best_accuracy'] for fr in fold_results_nomlm]):.4f} ± {np.std([fr['best_accuracy'] for fr in fold_results_nomlm]):.4f}")

In [13]:
with open('objects/last/rna_fold_results_mlm_last_l_conc.pkl', 'wb') as f:
    pickle.dump(fold_results, f)
with open('objects/last/rna_fold_results_nomlm_last_l_conc.pkl', 'wb') as f:
    pickle.dump(fold_results_nomlm, f)

with open('objects/last/rna_fold_results_mlm_last_l_conc.pkl', 'rb') as f:
    fold_results = pickle.load(f)
with open('objects/last/rna_fold_results_nomlm_last_l_conc.pkl', 'rb') as f:
        fold_results_nomlm = pickle.load(f)

In [ ]:
# compute mean and std of train_auc and best_val_auc keys in fold_results and fold_results_nomlm
train_auc_mlm = np.mean([fr['train_auc'] for fr in fold_results])
train_auc_nomlm = np.mean([fr['train_auc'] for fr in fold_results_nomlm])
train_auc_mlm_std = np.std([fr['train_auc'] for fr in fold_results])
train_auc_nomlm_std = np.std([fr['train_auc'] for fr in fold_results_nomlm])
best_val_auc_mlm = np.mean([fr['best_val_auc'] for fr in fold_results])
best_val_auc_nomlm = np.mean([fr['best_val_auc'] for fr in fold_results_nomlm])
best_val_auc_mlm_std = np.std([fr['best_val_auc'] for fr in fold_results])
best_val_auc_nomlm_std = np.std([fr['best_val_auc'] for fr in fold_results_nomlm])

print(f"Train AUC: MLM: {train_auc_mlm:.4f} ± {train_auc_mlm_std:.4f}, No-MLM: {train_auc_nomlm:.4f} ± {train_auc_nomlm_std:.4f}")
print(f"Best Val AUC: MLM: {best_val_auc_mlm:.4f} ± {best_val_auc_mlm_std:.4f}, No-MLM: {best_val_auc_nomlm:.4f} ± {best_val_auc_nomlm_std:.4f}")

In [ ]:
# print each fold best val auc separate based on mlm and no mlm 5 folds for each)
for i in range(5):
    print(f"Fold {i+1}: MLM: {fold_results[i]['best_val_auc']:.4f}, No-MLM: {fold_results_nomlm[i]['best_val_auc']:.4f}")
print(f"val auc mlm: {np.mean([fr['best_val_auc'] for fr in fold_results]):.4f} ± {np.std([fr['best_val_auc'] for fr in fold_results]):.4f}")
print(f"val auc no mlm: {np.mean([fr['best_val_auc'] for fr in fold_results_nomlm]):.4f} ± {np.std([fr['best_val_auc'] for fr in fold_results_nomlm]):.4f}")
# compute p-value for mlm and no-mlm values for val auc
from scipy.stats import ttest_rel
print("p-value for val auc: ", ttest_rel([fr['best_val_auc'] for fr in fold_results], [fr['best_val_auc'] for fr in fold_results_nomlm]))

In [ ]:
aucs_stored, aucs, val_preds, val_labels = evaluate_cls_cv("RNA", fold_results, model_config, labelled_dataset, device)
print([f"({a:.4f} | {b:.4f})" for a,b in zip(aucs_stored, aucs)])
plot_roc_auc_curve(val_preds, val_labels, m_type="RNA", aggregate=False)